# Etapa 7 — Variáveis associadas ao gênero (SHAP)

Projeto SME0828 — Introdução à Ciência de Dados, ICMC/USP.

Semente do código: 444. Reexecute o notebook para atualizar os resultados exibidos.


O modelo sem gênero ainda prevê taxas de negação diferentes entre mulheres e homens. Nesta etapa, testamos quanto as demais variáveis permitem prever o gênero e quais delas mais influenciam essa previsão e a decisão de crédito.

### 1. SHAP

O SHAP mede quanto cada variável contribui para uma previsão. Para modelos de árvores, usamos `shap.TreeExplainer`.

In [1]:
from pathlib import Path
import os

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abra este notebook dentro do reposit?rio, que deve conter a pasta data/")
os.chdir(ROOT)

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")

SEMENTE = 444
random.seed(SEMENTE)
np.random.seed(SEMENTE)

BLUE   = "#2a78d6"
ORANGE = "#eb6834"
GRAY   = "#898781"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
os.makedirs("figuras", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

ModuleNotFoundError: No module named 'shap'

### 2. Preparação dos dados

Usamos a mesma amostra, preparação, divisão de treino e teste e semente do notebook `05_modelagem_h2`.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "code" / "scripts"))
from preparo_hmda import carregar_amostra_modelagem

data = carregar_amostra_modelagem()
print("N?mero de linhas e colunas:", data.shape)


### 3. Codificação das variáveis categóricas

O SHAP precisa de categorias numéricas neste modelo. Codificamos DTI e idade em ordem crescente; os códigos de estado e tipo de empréstimo são arbitrários. As árvores podem separar esses valores em várias divisões.

In [ ]:
ordem_dti = ["<20%", "20%-<30%", "30%-<36%", "36%-<50%", "50%-60%", ">60%"]
ordem_idade = ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"]

data["dti"] = data["dti"].astype(str).map({faixa: i for i, faixa in enumerate(ordem_dti)})
data["applicant_age"] = data["applicant_age"].map({faixa: i for i, faixa in enumerate(ordem_idade)})
data["loan_type"] = data["loan_type"].astype(int)

estados = sorted(data["state_code"].unique())
data["state_code"] = data["state_code"].map({uf: i for i, uf in enumerate(estados)})

atributos = ["log_renda", "ltv", "dti", "loan_type", "state_code", "applicant_age"]
nomes = {"log_renda": "Renda (log)", "ltv": "LTV", "dti": "DTI", "loan_type": "Tipo de empréstimo",
         "state_code": "Estado", "applicant_age": "Idade"}

data[atributos].head()

,log_renda,ltv,dti,loan_type,state_code,applicant_age
0,4.174387,97.000,3,1,45,1
1,5.181784,80.000,3,1,4,2
2,4.110874,96.500,4,2,4,0
3,4.234107,97.000,1,1,24,1
4,4.077537,98.189,1,2,23,0


A divisão de treino e teste é a mesma do notebook `05_modelagem_h2`.

In [ ]:
estrato = data["negado"].astype(str) + "_" + data["mulher"].astype(str)
treino, teste = train_test_split(data, test_size=0.3, stratify=estrato, random_state=SEMENTE)

print("Treino:", treino.shape)
print("Teste: ", teste.shape)

Treino: (1073019, 10)
Teste:  (459866, 10)


### 4. Prever gênero a partir das demais variáveis

Treinamos um Gradient Boosting para prever `mulher` usando os atributos do modelo de crédito sem gênero. AUC = 0,5 corresponde ao acaso; valores maiores indicam informação preditiva sobre o gênero.

In [ ]:
classificador_genero = HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, max_iter=300,
                                                      early_stopping=True, random_state=SEMENTE)
classificador_genero.fit(treino[atributos], treino["mulher"])

proba_mulher = classificador_genero.predict_proba(teste[atributos])[:, 1]
auc_genero = roc_auc_score(teste["mulher"], proba_mulher)
print(f"AUC para prever o gênero: {auc_genero:.4f}")

AUC para prever o gênero: 0.6642


Calculamos o intervalo de confiança da AUC por bootstrap, com mil reamostragens do teste.

In [ ]:
y_genero = teste["mulher"].values
n = len(y_genero)
aucs = []
for b in range(1000):
    idx = np.random.randint(0, n, n)
    aucs.append(roc_auc_score(y_genero[idx], proba_mulher[idx]))

ic_inf, ic_sup = np.percentile(aucs, [2.5, 97.5])
print(f"AUC = {auc_genero:.4f}  (IC 95%: {ic_inf:.4f} a {ic_sup:.4f})")

AUC = 0.6642  (IC 95%: 0.6627 a 0.6658)


A diagonal representa o desempenho esperado ao acaso.

In [ ]:
fpr, tpr, _ = roc_curve(teste["mulher"], proba_mulher)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(fpr, tpr, color=ORANGE, linewidth=2.5, label=f"Classificador de gênero (AUC = {auc_genero:.3f})")
ax.plot([0, 1], [0, 1], color=GRAY, linestyle="--", linewidth=1.5, label="Aleatório (AUC = 0,5)")
ax.set_xlabel("Taxa de Falsos Positivos")
ax.set_ylabel("Taxa de Verdadeiros Positivos")
ax.set_title("Prevendo o gênero a partir das outras variáveis", fontsize=12, fontweight="bold")
ax.legend(frameon=False, loc="lower right")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/etapa_07_proxies_roc_genero.png", dpi=300)
plt.show()

### 5. Importância das variáveis

A importância SHAP é a média do valor absoluto das contribuições de cada variável. Calculamos os valores em uma amostra de 50 mil pedidos do teste.

In [ ]:
amostra = teste.sample(n=50_000, random_state=SEMENTE)

explicador_genero = shap.TreeExplainer(classificador_genero)
shap_genero = explicador_genero(amostra[atributos])

importancia_genero = pd.Series(np.abs(shap_genero.values).mean(axis=0), index=atributos)
importancia_genero = importancia_genero.sort_values(ascending=False).rename(index=nomes)
importancia_genero.round(4)

Renda (log)           0.2546
Idade                 0.2468
Tipo de empréstimo    0.1986
Estado                0.0763
LTV                   0.0559
DTI                   0.0495
dtype: float64

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importancia_genero.index[::-1], importancia_genero.values[::-1], color=ORANGE, alpha=0.85)
ax.set_xlabel("Importância (média do |valor SHAP|)")
ax.set_title("Variáveis que mais denunciam o gênero", fontsize=13, fontweight="bold")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/etapa_07_proxies_importancia_shap.png", dpi=300)
plt.show()

O gráfico beeswarm mostra cada pedido. A posição horizontal indica a contribuição para prever gênero feminino; a cor representa o valor da variável. Por exemplo, valores baixos de renda à direita indicam associação com a previsão de mulher.

In [ ]:
shap_genero.feature_names = [nomes[a] for a in atributos]
shap.plots.beeswarm(shap_genero, show=False)
plt.title("SHAP do classificador de gênero", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figuras/etapa_07_proxies_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

### 6. Diferenças entre os grupos

Comparamos as variáveis mais importantes entre mulheres e homens na amostra.

In [ ]:
comparacao = pd.DataFrame({
    "Renda mediana (mil US$)": data.groupby("mulher")["log_renda"].median().map(np.exp),
    "Idade: até 34 anos (%)": data.groupby("mulher")["applicant_age"].apply(lambda x: 100 * (x <= 1).mean()),
    "Idade: 55 anos ou mais (%)": data.groupby("mulher")["applicant_age"].apply(lambda x: 100 * (x >= 4).mean()),
    "Empréstimo FHA (%)": data.groupby("mulher")["loan_type"].apply(lambda x: 100 * (x == 2).mean()),
    "Empréstimo VA (%)": data.groupby("mulher")["loan_type"].apply(lambda x: 100 * (x == 3).mean()),
    "LTV mediano": data.groupby("mulher")["ltv"].median(),
    "DTI 50% ou mais (%)": data.groupby("mulher")["dti"].apply(lambda x: 100 * (x >= 4).mean()),
}).T
comparacao.columns = ["Homens", "Mulheres"]
comparacao.round(1)

,Homens,Mulheres
Renda mediana (mil US$),95.0,82.0
Idade: até 34 anos (%),42.8,32.9
Idade: 55 anos ou mais (%),14.6,22.3
Empréstimo FHA (%),22.3,27.3
Empréstimo VA (%),12.7,4.5
LTV mediano,95.0,93.3
DTI 50% ou mais (%),13.9,15.1


Tipos de empréstimo: 1 = convencional, 2 = FHA, 3 = VA e 4 = USDA.

### 7. Variáveis associadas à negação

Agora avaliamos quais variáveis contribuem para a diferença nas previsões de negação. Retreinamos o modelo de crédito sem gênero com a mesma divisão e os mesmos hiperparâmetros do notebook `05_modelagem_h2`.

In [ ]:
modelo_credito = HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, max_iter=300,
                                                early_stopping=True, random_state=SEMENTE)
modelo_credito.fit(treino[atributos], treino["negado"])

auc_credito = roc_auc_score(teste["negado"], modelo_credito.predict_proba(teste[atributos])[:, 1])
print(f"AUC do modelo de crédito sem gênero: {auc_credito:.4f}  (notebook 05: 0,7867)")

AUC do modelo de crédito sem gênero: 0.7848  (notebook 05: 0,7867)


Os valores SHAP somam a previsão em escala de log-odds. Comparamos a contribuição média de cada variável entre mulheres e homens. Valores positivos aumentam a previsão de negação para mulheres; valores negativos a reduzem.

In [ ]:
explicador_credito = shap.TreeExplainer(modelo_credito)
shap_credito = explicador_credito(amostra[atributos])

eh_mulher = amostra["mulher"].values == 1
contribuicao = pd.Series(
    shap_credito.values[eh_mulher].mean(axis=0) - shap_credito.values[~eh_mulher].mean(axis=0),
    index=atributos).rename(index=nomes).sort_values()

print(f"Diferença total na previsão (log-odds), mulheres - homens: {contribuicao.sum():.4f}")
contribuicao.round(4)

Diferença total na previsão (log-odds), mulheres - homens: 0.1402


LTV                  -0.0129
Estado                0.0002
DTI                   0.0174
Idade                 0.0207
Tipo de empréstimo    0.0419
Renda (log)           0.0729
dtype: float64

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cores = [ORANGE if v > 0 else BLUE for v in contribuicao.values]
ax.barh(contribuicao.index, contribuicao.values, color=cores, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Contribuição para a diferença (log-odds)\n← reduz negação de mulheres | aumenta negação de mulheres →")
ax.set_title("O que faz o modelo negar mais as mulheres", fontsize=13, fontweight="bold")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("figuras/etapa_07_proxies_contribuicao_negacao.png", dpi=300)
plt.show()

### 8. Salvamento dos resultados

In [ ]:
tabela = pd.DataFrame({
    "Importância para prever o gênero (SHAP)": importancia_genero,
    "Contribuição para a diferença na negação (log-odds)": contribuicao,
}).sort_values("Importância para prever o gênero (SHAP)", ascending=False)

with open("resultados/etapa_07_proxies_shap.md", "w", encoding="utf-8") as f:
    f.write(f"Classificador de gênero: AUC = {auc_genero:.4f} (IC 95%: {ic_inf:.4f} a {ic_sup:.4f}), "
            f"teste com {len(teste):,} pedidos.\n\n")
    f.write(tabela.round(4).to_markdown())
    f.write("\n\nImportância: média do |SHAP| do classificador de gênero (amostra de 50 mil pedidos do teste). "
            "Contribuição: média do SHAP do modelo de crédito sem gênero entre mulheres menos entre homens.\n")
tabela.round(4)

,Importância para prever o gênero (SHAP),Contribuição para a diferença na negação (log-odds)
Renda (log),0.2546,0.0729
Idade,0.2468,0.0207
Tipo de empréstimo,0.1986,0.0419
Estado,0.0763,0.0002
LTV,0.0559,-0.0129
DTI,0.0495,0.0174


### Conclusão

- O classificador prevê gênero com AUC de 0,664 (IC 95%: 0,663 a 0,666), acima do acaso. As variáveis carregam informação sobre o gênero, mas não o determinam.
- Os principais atributos associados à previsão de gênero são renda, idade e tipo de empréstimo. Mulheres têm renda mediana menor e são, em média, mais velhas; empréstimos VA são mais comuns entre homens e FHA entre mulheres.
- Estado, LTV e DTI têm menor importância para prever gênero.
- A renda e o tipo de empréstimo explicam a maior parte da diferença de negação prevista; idade e DTI também contribuem. O LTV atua na direção oposta.
- O modelo aprende a associação entre empréstimo VA e aprovação. Como esse tipo é mais comum entre homens, pode contribuir para a diferença mesmo sem a variável gênero.

As variáveis podem funcionar como proxies, mas a análise não demonstra discriminação. A codificação numérica usada para o SHAP produziu AUC de 0,785, próxima à do modelo original (0,787).